# Supplementary Table 5

Individual-function Spearman correlations between microbial DNA, RNA and protein layers across 77 PREDICT patients: 30,581 function-by-layer-pair tests in nine Benjamini-Hochberg families (three functional representations by three layer pairs), the nine-family summary, the eligibility of every candidate function, and the comparison with the previous functional allocation. This notebook rebuilds the workbook from `source_data/STab5/`, compares it cell by cell with the table as published, recomputes every test from the patient-level profiles and prints the numbers quoted in the text.

Run from the repository root: `python paper.py build STab5`.

In [1]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

## Table

Five sheets: `Read_me`, `All_30581_tests` (one row per eligible function and layer pair: patients, Spearman rho, two-sided P, Benjamini-Hochberg q within the family), `Nine_family_summary`, `All_eligibility` (184,386 candidate function-by-layer-pair rows) and `Eligibility_common` (functions eligible under both the previous and the revised allocation). Supports "Supplementary Table 5 | Individual-function correlations between DNA, RNA and protein layers." (v28d line 483) and the sentences on v28d lines 141 and 829. n = 77 patients; unit of each test: patient. The workbook is written by the producer's code and then edited as in the Source Data package (Read_me sheet replaced, `Eligibility_common` columns renamed). Its cell values must equal `expected/Supplementary_Table_5.xlsx`; the two support files must be byte-identical to the package copies.

In [2]:
import hashlib, json
import pandas as pd
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
d = panels.load(SD)
xlsx, matched, ver = panels.build_table(SD, OUT)
elig_bytes, summary_bytes = panels.package_support(matched, ver)
(OUT / "eligibility_matched_correlations.tsv").write_bytes(elig_bytes)
(OUT / "common_eligibility_summary.tsv").write_bytes(summary_bytes)
exp = UNIT / "expected"
table_receipt = {
    "xlsx": panels.compare_xlsx_values(xlsx, exp / "Supplementary_Table_5.xlsx"),
    "eligibility_matched_correlations.tsv": {"new_sha256": sha(OUT / "eligibility_matched_correlations.tsv"),
                                             "expected_sha256": sha(exp / "eligibility_matched_correlations.tsv")},
    "common_eligibility_summary.tsv": {"new_sha256": sha(OUT / "common_eligibility_summary.tsv"),
                                       "expected_sha256": sha(exp / "common_eligibility_summary.tsv")},
}
table_receipt["xlsx"]["expected_sha256"] = sha(exp / "Supplementary_Table_5.xlsx")
for k in ("eligibility_matched_correlations.tsv", "common_eligibility_summary.tsv"):
    table_receipt[k]["identical"] = table_receipt[k]["new_sha256"] == table_receipt[k]["expected_sha256"]
table_receipt["result"] = "IDENTICAL" if all(v["identical"] for v in table_receipt.values()) else "DIFFERENT"
print(json.dumps({k: (v if k == "result" else v["identical"]) for k, v in table_receipt.items()}, indent=1))
print("cells compared:", table_receipt["xlsx"]["cells"], "exactly equal:", table_receipt["xlsx"]["exact_cells"])
pd.read_csv(OUT / "common_eligibility_summary.tsv", sep="\t")

{
 "xlsx": true,
 "eligibility_matched_correlations.tsv": true,
 "common_eligibility_summary.tsv": true,
 "result": "IDENTICAL"
}
cells compared: 1351138 exactly equal: 1351138


,level,layer_a,layer_b,n_common,n_newly_eligible,n_lost,median_rho_revised_common,median_rho_previous_common,median_rho_revised_newly_eligible
0,KO,metaG,metaT,3995,0,0,0.351885,0.351885,NaN
1,KO,metaG,microbial_metaP,1912,2872,0,0.086063,0.100677,0.015003
2,KO,metaT,microbial_metaP,1797,2017,0,0.089936,0.100000,0.013642
3,OG,metaG,metaT,5796,0,0,0.359114,0.359114,NaN
4,OG,metaG,microbial_metaP,1918,4973,0,0.079013,0.100845,0.012195
5,OG,metaT,microbial_metaP,1803,2720,0,0.098756,0.099286,0.016528
6,Pathway,metaG,metaT,249,0,0,0.239865,0.239865,NaN
7,Pathway,metaG,microbial_metaP,223,57,0,0.066802,0.105789,-0.034256
8,Pathway,metaT,microbial_metaP,221,28,0,0.109364,0.124192,-0.055170


## Recomputation of the 30,581 tests

For each representation (OG, KO, pathway) and each pair of layers, profiles of the 77 patients are normalized to relative signal over the function identifiers shared by that pair; each function with positive values in both layers in at least 35 patients gets one Spearman correlation (two-sided P, `scipy.stats.spearmanr`); P values are adjusted by Benjamini-Hochberg within each of the nine families (Methods, v28d line 829). The recomputed tests, eligibility table and summary are compared with the tables as produced (`source_data/STab5/`), and the recomputed test table is written in the producer's format and compared byte for byte.

In [3]:
tests, eligibility, summary = panels.stat_individual_functions(SD)
K = ["level", "layer_a", "layer_b", "feature"]
recompute_receipt = {
    "tests": panels.compare_frames(tests, d["tests"], K, ["rho", "p", "q"], ["n_patients"]),
    "eligibility": panels.compare_frames(eligibility, d["eligibility"], K, [], ["n_patients", "eligible"]),
    "summary": panels.compare_frames(summary, d["summary"], K[:3], ["median_rho"], ["features", "significant"]),
}
tests.to_csv(OUT / "recomputed_tests.tsv", sep="\t", index=False)
recompute_receipt["tests_tsv_byte_identical"] = sha(OUT / "recomputed_tests.tsv") == sha(SD / "crossomics_same_function_correlations.tsv")
ok = (all(v["keys_equal"] and all(x for k, x in v.items() if k.startswith("equal_")) for k, v in recompute_receipt.items() if isinstance(v, dict))
      and all(x <= 1e-12 for v in recompute_receipt.values() if isinstance(v, dict) for k, x in v.items() if k.startswith("max_abs_diff")))
recompute_receipt["result"] = "MATCH" if ok else "DIFFERENT"
print(json.dumps(recompute_receipt, indent=1))
summary

{
 "tests": {
  "rows_new": 30581,
  "rows_old": 30581,
  "keys_equal": true,
  "max_abs_diff_rho": 9.974659986866641e-17,
  "max_abs_diff_p": 1.1102230246251565e-16,
  "max_abs_diff_q": 1.1102230246251565e-16,
  "equal_n_patients": true
 },
 "eligibility": {
  "rows_new": 184386,
  "rows_old": 184386,
  "keys_equal": true,
  "equal_n_patients": true,
  "equal_eligible": true
 },
 "summary": {
  "rows_new": 9,
  "rows_old": 9,
  "keys_equal": true,
  "max_abs_diff_median_rho": 9.71445146547012e-17,
  "equal_features": true,
  "equal_significant": true
 },
 "tests_tsv_byte_identical": true,
 "result": "MATCH"
}


,level,layer_a,layer_b,features,median_rho,significant
0,KO,metaG,metaT,3995,0.351885,2816
1,KO,metaG,microbial_metaP,4784,0.041214,163
2,KO,metaT,microbial_metaP,3814,0.048012,101
3,OG,metaG,metaT,5796,0.359114,3954
4,OG,metaG,microbial_metaP,6891,0.028956,138
5,OG,metaT,microbial_metaP,4523,0.046053,79
6,Pathway,metaG,metaT,249,0.239865,115
7,Pathway,metaG,microbial_metaP,280,0.049332,9
8,Pathway,metaT,microbial_metaP,249,0.083574,18


## Statistics

One row per Benjamini-Hochberg family: family size, patients per test, median and interquartile range of rho, tests with q < 0.05, smallest P and q. These fill `STATS.tsv`.

In [4]:
families = panels.stat_families(d["tests"])
pd.set_option("display.precision", 17)
families

,level,layer_a,layer_b,family_size,n_min,n_max,median_rho,rho_q25,rho_q75,significant,min_p,min_q
0,KO,metaG,metaT,3995,35,77,0.35188495714811491,0.23827744031318998,0.46196520737524405,2816,0.00000000000000024,0.00000000000094661
1,KO,metaG,microbial_metaP,4784,35,77,0.04121380752959695,-0.05826162842204653,0.15785996897839000,163,0.00000000565820550,0.00001401931588583
2,KO,metaT,microbial_metaP,3814,35,77,0.04801174058139999,-0.05285289891268147,0.15662625109033296,101,0.00000002515278294,0.00006347350683572
3,OG,metaG,metaT,5796,35,77,0.35911436344872871,0.23881381776118610,0.46790285519664931,3954,0.00000000000000299,0.00000000001733257
4,OG,metaG,microbial_metaP,6891,35,77,0.02895558223289310,-0.07427480976023790,0.13589867305037578,138,0.00000000034231041,0.00000235886105154
5,OG,metaT,microbial_metaP,4523,35,77,0.04605315667651050,-0.05265432488784985,0.15337366722129026,79,0.00000000051903095,0.00000234757697058
6,Pathway,metaG,metaT,249,38,77,0.23986539776013449,0.13121089436878910,0.38424733161575259,115,0.00000000067693049,0.00000016855569264
7,Pathway,metaG,microbial_metaP,280,36,77,0.04933224670066770,-0.06354472595074093,0.17038224932961771,9,0.00000001499968455,0.00000419991167504
8,Pathway,metaT,microbial_metaP,249,38,77,0.08357432041642560,-0.00738734949261260,0.18718123981281870,18,0.00003721815844364,0.00805373832684530


## Checks against the table as produced and recomputed numbers

Writes the receipts `checks/table.json` and `checks/recompute.json`, then prints every number in `NUMBERS.tsv` order at full precision. There are no figure panels in this unit, so there are no pixel checks.

In [5]:
(UNIT / "checks").mkdir(exist_ok=True)
(UNIT / "checks" / "table.json").write_text(json.dumps(table_receipt, indent=1) + "\n")
(UNIT / "checks" / "recompute.json").write_text(json.dumps(recompute_receipt, indent=1) + "\n")
print("table:", table_receipt["result"], "| recompute:", recompute_receipt["result"])
assert table_receipt["result"] == "IDENTICAL" and recompute_receipt["result"] == "MATCH"
for label, value in panels.numbers(summary, ver, tests, d["previous"]):
    print(f"{label}\t{value!r}")

table: IDENTICAL | recompute: MATCH
L141 DNA/RNA-protein median rho, min	0.02895558223289316
L141 DNA/RNA-protein median rho, max	0.08357432041642567
L141 DNA-RNA median rho, min	0.23986539776013455
L141 DNA-RNA median rho, max	0.3591143634487288
L141 common-eligible protein median rho (revised), min	0.0668016194331983
L141 common-eligible protein median rho (revised), max	0.1093643198906356
L829 smallest number of patients in any test (rule: at least MIN_PATIENTS)	35
L829 BH families	'nine'
L829 tests under the revised allocation	30581
L829 common-eligible protein median rho (previous), min	0.099286177806619
L829 common-eligible protein median rho (previous), max	0.1241915978758084
L893 BH families for individual-function correlations	'nine'
